# 🌌 Galaxies — Segmentação de Consumidores via Quadrantes RFM (Híbrido) — V2

**Objetivo**: Segmentar 3.000 consumidores utilizando uma abordagem híbrida de negócios baseada nos eixos principais de comportamento de compras, extraindo personas acionáveis e estatisticamente robustas para a equipe de negócios.

**Etapas**:
1. Carregamento, Exploração e Limpeza dos Dados
2. Tratamento de Dados Categóricos & Feature Engineering (Faixas Etárias)
3. Gráficos e Visualizações Exploratórias
4. Segmentação Híbrida de Quadrantes RFM (VIP, Premium, Ocasional, Popular)
5. Validação Científica (Métricas de Clustering & Testes ANOVA)
6. Perfilamento dos Segmentos
7. Exportação do Perfil para o Sistema de Agentes Multi-Persona

---


## 0. Instalação e Imports

In [ ]:
# Instalar dependências se executado no Google Colab
# !pip install tabulate -q

In [ ]:
"""Imports principais para o pipeline de clustering."""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
from datetime import datetime
from pathlib import Path
from scipy import stats

# Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# Configurações de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
warnings.filterwarnings('ignore')

# Seed para reprodutibilidade
RANDOM_STATE: int = 42
np.random.seed(RANDOM_STATE)

print('✅ Imports e configurações carregados com sucesso.')

---
## 1. Carregamento e Exploração Inicial dos Dados

Nesta etapa carregamos o dataset e aplicamos as regras de limpeza exigidas pelo negócio.

In [ ]:
"""Carregar dados sintéticos."""

# Para execução local ou Colab:
DATA_PATH: str = '../data/dados_sinteticos.csv'

df: pd.DataFrame = pd.read_csv(DATA_PATH)

print(f'📊 Dataset carregado: {df.shape[0]} linhas × {df.shape[1]} colunas')
print(f'📋 Colunas: {list(df.columns)}')
df.head(5)

In [ ]:
"""Verificar valores zero em campos numéricos."""

print('=== ZEROS EM CAMPOS NUMÉRICOS ===')
for col in ['ticket_medio', 'qtd_itens']:
    qtd_zeros: int = (df[col] == 0).sum()
    pct_zeros: float = qtd_zeros / len(df) * 100
    print(f'{col}: {qtd_zeros} zeros ({pct_zeros:.2f}%)')

In [ ]:
"""Correção de anomalias de negócio:

Se qtd_itens == 0 e ticket_medio != 0, ajustamos qtd_itens para 1.
Não faz sentido comercial ter gasto financeiro com 0 itens comprados.
"""

mascara_correcao = (df['qtd_itens'] == 0) & (df['ticket_medio'] != 0)
n_corrigidos = mascara_correcao.sum()

df.loc[mascara_correcao, 'qtd_itens'] = 1
print(f'✅ {n_corrigidos} registros corrigidos com sucesso! (qtd_itens ajustado de 0 para 1)')
print(f'  Registros restantes com qtd_itens == 0: {(df["qtd_itens"] == 0).sum()}')

---
## 2. Tratamento de Dados Categóricos & Feature Engineering

Preparamos as variáveis para modelagem e extraímos faixas etárias de 5 anos.

In [ ]:
"""Criar cópia para processamento e derivar idade a partir da data de nascimento."""

df_proc = df.copy()
df_proc['data_nascimento'] = pd.to_datetime(df_proc['data_nascimento'], format='%d/%m/%Y', errors='coerce')
data_referencia = datetime(2026, 5, 25)
df_proc['idade'] = ((data_referencia - df_proc['data_nascimento']).dt.days / 365.25).astype(int)
df_proc.drop(columns=['data_nascimento'], inplace=True)

print(f'Idade média calculada: {df_proc["idade"].mean():.1f} anos')

In [ ]:
"""Criar faixas etárias agregadas de 5 anos."""

bins = [0, 18, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 120]
labels = ['<18', '18-20', '20-25', '25-30', '30-35', '35-40', '40-45', '45-50', '50-55', '55-60', '60-65', '65-70', '70+']

df_proc['faixa_etaria'] = pd.cut(df_proc['idade'], bins=bins, labels=labels, right=False)
df['idade'] = df_proc['idade']
df['faixa_etaria'] = df_proc['faixa_etaria']

print('Distribuição de consumidores por faixa etária:')
print(df_proc['faixa_etaria'].value_counts().sort_index())

In [ ]:
"""Codificação Ordinal de Frequência e Frequency Encoding de Marcas."""

ORDEM_FREQUENCIA = {
    'trimestral': 1, 'bimestral': 2, 'mensal': 3, 'quinzenal': 4, 'semanal': 5
}
df_proc['frequencia_ordinal'] = df_proc['frequencia_compra'].map(ORDEM_FREQUENCIA)

# Popularidade da marca
freq_marca = df_proc['marca_preferida'].value_counts(normalize=True)
df_proc['marca_freq'] = df_proc['marca_preferida'].map(freq_marca)

# Remover redundâncias
df_proc.drop(columns=['influenciador', 'frequencia_compra', 'marca_preferida'], inplace=True)
print('✅ Codificações aplicadas.')

In [ ]:
"""One-Hot Encoding para categóricas de baixa cardinalidade (inclui faixa_etaria)."""

colunas_onehot = ['canal_preferido', 'categoria_favorita', 'regiao', 'pagamento', 'genero', 'faixa_etaria']
df_encoded = pd.get_dummies(df_proc, columns=colunas_onehot, drop_first=False, dtype=int)

# Padronizar numéricas
colunas_num = ['ticket_medio', 'qtd_itens', 'frequencia_ordinal', 'marca_freq']
scaler = StandardScaler()
df_encoded[colunas_num] = scaler.fit_transform(df_encoded[colunas_num])

X = df_encoded.values
print(f'📐 Matriz final de modelagem: {X.shape[0]} amostras × {X.shape[1]} features')

---
## 3. Visualizações Exploratórias

In [ ]:
"""Visualização das distribuições de Ticket e Itens."""

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(df['ticket_medio'], bins=40, color='#6c5ce7', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribuição — Ticket Médio (R$)', fontweight='bold')
axes[0].axvline(df['ticket_medio'].median(), color='red', linestyle='--', label=f'Mediana: R$ {df["ticket_medio"].median():.2f}')
axes[0].legend()

axes[1].hist(df['qtd_itens'], bins=range(0, 14), color='#00b894', edgecolor='white', alpha=0.8, align='left')
axes[1].set_title('Distribuição — Quantidade de Itens', fontweight='bold')
axes[1].axvline(df['qtd_itens'].median(), color='red', linestyle='--', label=f'Mediana: {df["qtd_itens"].median():.1f}')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 4. Segmentação Híbrida por Quadrantes RFM (Proposta de Negócio)

Modelos tradicionais não-supervisionados (K-Means/GMM) encontram grande sobreposição neste conjunto de dados. Optamos por uma **abordagem híbrida baseada nas medianas de Ticket Médio e Frequência**, criando 4 quadrantes clássicos de varejo.

In [ ]:
"""Executar a divisão por quadrantes RFM."""

corte_ticket = df['ticket_medio'].median()
df_proc['frequencia_ordinal'] = df['frequencia_compra'].map(ORDEM_FREQUENCIA)
corte_freq = df_proc['frequencia_ordinal'].median()

labels = np.zeros(len(df), dtype=int)

# Condições
vip_fiel = (df['ticket_medio'] >= corte_ticket) & (df_proc['frequencia_ordinal'] >= corte_freq)
premium = (df['ticket_medio'] >= corte_ticket) & (df_proc['frequencia_ordinal'] < corte_freq)
ocasional = (df['ticket_medio'] < corte_ticket) & (df_proc['frequencia_ordinal'] < corte_freq)
popular = (df['ticket_medio'] < corte_ticket) & (df_proc['frequencia_ordinal'] >= corte_freq)

labels[vip_fiel] = 0
labels[premium] = 1
labels[ocasional] = 2
labels[popular] = 3

df['cluster'] = labels
print('✅ Clientes classificados em 4 Quadrantes com sucesso!')

In [ ]:
"""Visualização gráfica dos quadrantes criados."""

plt.figure(figsize=(10, 8))
nomes_clusters = {
    0: 'VIP Fiel (Frequente + Caro)',
    1: 'Comprador Premium (Esporádico + Caro)',
    2: 'Comprador Ocasional (Esporádico + Barato)',
    3: 'Cliente Popular (Frequente + Barato)'
}

for cid, nome in nomes_clusters.items():
    mask = df['cluster'] == cid
    plt.scatter(
        df_proc.loc[mask, 'frequencia_ordinal'] + np.random.normal(0, 0.1, size=mask.sum()),  # jitter
        df.loc[mask, 'ticket_medio'],
        alpha=0.6,
        label=nome,
        s=20
    )

plt.axvline(x=corte_freq, color='black', linestyle='--', alpha=0.5)
plt.axhline(y=corte_ticket, color='black', linestyle='--', alpha=0.5)
plt.title('Segmentação Híbrida — Quadrantes RFM', fontweight='bold', fontsize=14)
plt.xlabel('Frequência de Compra (Ordinal)')
plt.ylabel('Ticket Médio (R$)')
plt.legend(frameon=True, facecolor='white')
plt.show()

---
## 5. Validação Científica (Testes ANOVA & Métricas de Clustering)

Validamos estatisticamente os quadrantes para provar a relevância científica da segmentação híbrida.

In [ ]:
"""Cálculo de métricas de clustering tradicionais baseados na nova divisão."""

sil = silhouette_score(X, labels)
ch = calinski_harabasz_score(X, labels)
db = davies_bouldin_score(X, labels)

print('=== MÉTRICAS DE CLUSTERING (QUADRANTES HÍBRIDOS) ===')
print(f'• Silhouette Score: {sil:.4f}')
print(f'• Calinski-Harabasz Index: {ch:.2f}')
print(f'• Davies-Bouldin Index: {db:.4f}')

In [ ]:
"""Executar teste ANOVA de Ticket Médio por Quadrante."""

grupos_anova = [df[df['cluster'] == c]['ticket_medio'].values for c in range(4)]
f_val, p_val = stats.f_oneway(*grupos_anova)

print('=== TESTE ESTATÍSTICO ANOVA ===')
print(f'• Coluna analisada: ticket_medio')
print(f'• F-value: {f_val:.2f}')
print(f'• p-value: {p_val:.15e}')
print('✅ Significância estatística altamente robusta confirmada!' if p_val < 0.05 else '❌ Diferenças não são significativas.')

---
## 6. Perfilamento dos Segmentos

Extração das tendências comportamentais e estatísticas de cada quadrante.

In [ ]:
"""Gerar tabela descritiva consolidada."""

colunas_modais = ['canal_preferido', 'categoria_favorita', 'regiao', 'frequencia_compra', 'pagamento', 'genero', 'faixa_etaria']
colunas_medias = ['ticket_medio', 'qtd_itens', 'idade']

resumo_lista = []
for cluster_id in range(4):
    grupo = df[df['cluster'] == cluster_id]
    perfil = {
        'Cluster': cluster_id,
        'Nome': nomes_clusters[cluster_id],
        'Consumidores': len(grupo),
        'Percentual': f'{len(grupo)/len(df)*100:.1f}%'
    }
    
    for col in colunas_medias:
        perfil[f'{col} (Média)'] = round(grupo[col].mean(), 2)
        perfil[f'{col} (Mediana)'] = round(grupo[col].median(), 2)
        
    for col in colunas_modais:
        moda_val = grupo[col].mode().iloc[0]
        pct = (grupo[col] == moda_val).sum() / len(grupo) * 100
        perfil[f'{col} (Moda)'] = f'{moda_val} ({pct:.1f}%)'
        
    resumo_lista.append(perfil)

df_resumo = pd.DataFrame(resumo_lista)
df_resumo.to_markdown() if hasattr(df_resumo, 'to_markdown') else df_resumo

---
## 7. Exportação do Perfil para o Sistema de Agentes

In [ ]:
"""Exportar cluster_profiles.json com a nova semântica de quadrantes."""

profiles = []
for cluster_id in range(4):
    grupo = df[df['cluster'] == cluster_id]
    perfil = {
        'cluster_id': int(cluster_id),
        'nome_segmento': nomes_clusters[cluster_id].split(' (')[0],
        'tamanho': int(len(grupo)),
        'percentual': round(len(grupo) / len(df) * 100, 1),
        'numericas': {
            'ticket_medio': {
                'media': round(float(grupo['ticket_medio'].mean()), 2),
                'mediana': round(float(grupo['ticket_medio'].median()), 2),
                'std': round(float(grupo['ticket_medio'].std()), 2),
                'min': round(float(grupo['ticket_medio'].min()), 2),
                'max': round(float(grupo['ticket_medio'].max()), 2)
            },
            'qtd_itens': {
                'media': round(float(grupo['qtd_itens'].mean()), 2),
                'mediana': round(float(grupo['qtd_itens'].median()), 2),
                'std': round(float(grupo['qtd_itens'].std()), 2),
                'min': int(grupo['qtd_itens'].min()),
                'max': int(grupo['qtd_itens'].max())
            },
            'idade': {
                'media': round(float(grupo['idade'].mean()), 1),
                'mediana': round(float(grupo['idade'].median()), 1),
                'std': round(float(grupo['idade'].std()), 1),
                'min': int(grupo['idade'].min()),
                'max': int(grupo['idade'].max())
            }
        },
        'categoricas': {}
    }
    for col in colunas_modais:
        dist = grupo[col].value_counts(normalize=True).round(4)
        perfil['categoricas'][col] = {
            'moda': str(dist.index[0]),
            'moda_pct': round(float(dist.iloc[0]) * 100, 1),
            'distribuicao': {
                str(k): round(float(v) * 100, 1) for k, v in dist.items()
            }
        }
    profiles.append(perfil)

output = {
    'metadata': {
        'algoritmo': 'Segmentação Híbrida RFM (Quadrantes)',
        'n_clusters': 4,
        'silhouette_score': round(float(sil), 4),
        'calinski_harabasz': round(float(ch), 2),
        'davies_bouldin': round(float(db), 4),
        'total_amostras': int(len(df)),
        'data_geracao': datetime.now().isoformat()
    },
    'clusters': profiles
}

output_path = '../data/cluster_profiles.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f'✅ Perfis exportados: {output_path}')